# Feature Fusion Extraction — Motion (LK) + Visual (RF-DETR, DINOv2, VideoMAE)

Builds one fused feature vector per clip. **Branch-selectable** (see
`BRANCHES` in Step 0): run only the cheap branches (LK + RF-DETR --
no deep embedding models, fast even on CPU) or only the expensive ones
(DINOv2 + VideoMAE -- GPU strongly recommended) or all four. This
split exists for the branch ablation (`docs/findings.md`, 2026-09-19
entry) -- the first full run took 3.45 hours on CPU, almost entirely
DINOv2/VideoMAE forward passes; LK+RF-DETR alone should be dramatically
faster and doesn't need a GPU at all.

Output filename reflects which branches were run this session
(`features_<branches>.csv`, e.g. `features_lk_rfdetr.csv` or
`features_dinov2_videomae.csv` or `features_lk_rfdetr_dinov2_videomae.csv`
for all four) -- so different partial runs don't collide with each
other on Drive, and `ablation_fusion_branches.ipynb` can find whichever
combination(s) you've actually run.

**Requires:** both pull notebooks already run (again, if you deleted
the Drive data after the last run) --
`My Drive/CVPHG/xdviolence_riot/` and `My Drive/CVPHG/xdviolence_normal/`
each populated with their manifest CSV.

**Resumable:** re-running with the same `BRANCHES` selection skips any
`video_id` already present in that selection's output CSV.

## Step 0 — Setup and branch selection

**Edit `BRANCHES` below, then Runtime -> Change runtime type -> T4 GPU**
if it includes `'dinov2'` or `'videomae'` (recommended: run those two
on a T4 -- CPU is what made the full run take 3.45 hours last time).
`{'lk', 'rfdetr'}` alone runs fine on CPU (no GPU savings there --
LK is pure OpenCV, RF-DETR-Medium is small).

In [ ]:
!pip install rfdetr supervision opencv-python-headless transformers -q

In [ ]:
# Edit this set to choose what this run computes. Valid values:
# 'lk', 'rfdetr', 'dinov2', 'videomae'. Any subset is fine.
BRANCHES = {'lk', 'rfdetr'}

import csv
import os

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

RIOT_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_riot'
NORMAL_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_normal'
OUT_DIR = '/content/drive/MyDrive/CVPHG/fusion_features'
os.makedirs(OUT_DIR, exist_ok=True)

BRANCH_ORDER = ['lk', 'rfdetr', 'dinov2', 'videomae']
branch_suffix = '_'.join(b for b in BRANCH_ORDER if b in BRANCHES)
OUT_CSV = os.path.join(OUT_DIR, f'features_{branch_suffix}.csv')
print('This run computes:', sorted(BRANCHES))
print('Output file:', OUT_CSV)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if ('dinov2' in BRANCHES or 'videomae' in BRANCHES) and DEVICE == 'cpu':
    print('WARNING: DINOv2/VideoMAE selected but no GPU detected -- this will be slow. '
          'Runtime -> Change runtime type -> T4 GPU, then re-run this cell.')

In [ ]:
riot_manifest = pd.read_csv(os.path.join(RIOT_DIR, 'manifest_xdviolence_riot.csv'))
riot_manifest['clip_dir'] = RIOT_DIR
normal_manifest = pd.read_csv(os.path.join(NORMAL_DIR, 'manifest_xdviolence_normal.csv'))
normal_manifest['clip_dir'] = NORMAL_DIR

all_clips = pd.concat([riot_manifest, normal_manifest], ignore_index=True)
print(f"{len(riot_manifest)} riot + {len(normal_manifest)} normal = {len(all_clips)} clips total")

## Step 1 — Load only the models this run needs

In [ ]:
if 'rfdetr' in BRANCHES:
    from rfdetr import RFDETRMedium
    rfdetr_model = RFDETRMedium()
    PERSON_CLASS_ID = 0
    VEHICLE_CLASS_IDS = {1, 2, 3, 5, 7}  # bicycle, car, motorcycle, bus, truck
    DETECTION_THRESHOLD = 0.5

In [ ]:
if 'dinov2' in BRANCHES:
    dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
    dinov2_model.eval().to(DEVICE)

    DINOV2_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    DINOV2_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    def dinov2_embed_frame(frame_rgb: np.ndarray) -> np.ndarray:
        """frame_rgb: HxWx3 uint8. Returns a 768-d CLS embedding."""
        img = cv2.resize(frame_rgb, (224, 224), interpolation=cv2.INTER_AREA)
        tensor = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0) / 255.0
        tensor = (tensor - DINOV2_MEAN) / DINOV2_STD
        tensor = tensor.to(DEVICE)
        with torch.no_grad():
            feat = dinov2_model(tensor)  # (1, 768) pooled CLS embedding
        return feat.squeeze(0).cpu().numpy()

In [ ]:
if 'videomae' in BRANCHES:
    from transformers import VideoMAEModel, VideoMAEVideoProcessor
    videomae_processor = VideoMAEVideoProcessor.from_pretrained('MCG-NJU/videomae-base')
    videomae_model = VideoMAEModel.from_pretrained('MCG-NJU/videomae-base').eval().to(DEVICE)
    VIDEOMAE_NUM_FRAMES = 16  # required by this checkpoint's config (num_frames=16)

    def videomae_embed_clip(frames_16_rgb: list) -> np.ndarray:
        """frames_16_rgb: list of exactly 16 HxWx3 uint8 RGB frames.
        Returns a 768-d mean-pooled clip embedding."""
        inputs = videomae_processor(frames_16_rgb, return_tensors='pt')
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = videomae_model(**inputs)
        # last_hidden_state: (1, 1568, 768) -- mean-pool over all tokens for one clip vector
        return outputs.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()

## Step 2 — Motion branch: Lucas-Kanade optical flow (self-contained)

Mirrors `src/optical_flow.py` exactly (same params, same logic) since
this notebook doesn't clone the repo. Runs on a consecutive window of
frames (not evenly-spaced across the whole clip) -- LK's small-motion
assumption breaks down between widely separated frames.

In [ ]:
LK_FEATURE_PARAMS = dict(maxCorners=200, qualityLevel=0.01, minDistance=7, blockSize=7)
LK_WIN_SIZE = (15, 15)
LK_MAX_LEVEL = 2
LK_CRITERIA = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
LK_WINDOW_FRAMES = 32  # consecutive frames per clip, matches local pipeline's intent
LK_STRIDE = 2


def sample_consecutive_gray(video_path, max_frames=LK_WINDOW_FRAMES, stride=LK_STRIDE):
    cap = cv2.VideoCapture(video_path)
    frames = []
    idx = 0
    while len(frames) < max_frames:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        if idx % stride == 0:
            frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY))
        idx += 1
    cap.release()
    return frames


def compute_lk_summary(video_path):
    frames_gray = sample_consecutive_gray(video_path)
    if len(frames_gray) < 2:
        return dict(lk_mean_magnitude=0.0, lk_max_magnitude=0.0, lk_mean_angle_deg=0.0, lk_mean_tracked=0.0)

    prev_gray = frames_gray[0]
    prev_pts = cv2.goodFeaturesToTrack(prev_gray, mask=None, **LK_FEATURE_PARAMS)
    mags, max_mags, angles, tracked_counts = [], [], [], []

    for i in range(1, len(frames_gray)):
        curr_gray = frames_gray[i]
        if prev_pts is None or len(prev_pts) == 0:
            prev_pts = cv2.goodFeaturesToTrack(prev_gray, mask=None, **LK_FEATURE_PARAMS)
        if prev_pts is None or len(prev_pts) == 0:
            prev_gray = curr_gray
            continue

        next_pts, status, _ = cv2.calcOpticalFlowPyrLK(
            prev_gray, curr_gray, prev_pts, None,
            winSize=LK_WIN_SIZE, maxLevel=LK_MAX_LEVEL, criteria=LK_CRITERIA,
        )
        status = status.reshape(-1)
        good_prev = prev_pts.reshape(-1, 2)[status == 1]
        good_next = next_pts.reshape(-1, 2)[status == 1]

        if len(good_next) > 0:
            flow_vecs = good_next - good_prev
            m = np.linalg.norm(flow_vecs, axis=1)
            a = np.degrees(np.arctan2(flow_vecs[:, 1], flow_vecs[:, 0])) % 360
            mags.append(float(m.mean()))
            max_mags.append(float(m.max()))
            angles.append(float(a.mean()))
            tracked_counts.append(len(good_next))

        prev_gray = curr_gray
        prev_pts = good_next.reshape(-1, 1, 2).astype(np.float32) if len(good_next) > 0 else None

    if not mags:
        return dict(lk_mean_magnitude=0.0, lk_max_magnitude=0.0, lk_mean_angle_deg=0.0, lk_mean_tracked=0.0)
    return dict(
        lk_mean_magnitude=float(np.mean(mags)), lk_max_magnitude=float(np.max(max_mags)),
        lk_mean_angle_deg=float(np.mean(angles)), lk_mean_tracked=float(np.mean(tracked_counts)),
    )

## Step 3 — Visual branch helpers

In [ ]:
N_SPARSE_FRAMES = 4  # for RF-DETR + DINOv2 (evenly spaced across the clip)


def sample_frames_rgb(video_path, n):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []
    idxs = np.linspace(0, total - 1, min(n, total)).astype(int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame_bgr = cap.read()
        if ok:
            frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def compute_rfdetr_summary(frames_rgb):
    if not frames_rgb:
        return dict(rfdetr_mean_person=0.0, rfdetr_max_person=0.0,
                    rfdetr_mean_vehicle=0.0, rfdetr_mean_confidence=0.0)
    persons, vehicles, confs = [], [], []
    for frame in frames_rgb:
        det = rfdetr_model.predict(Image.fromarray(frame), threshold=DETECTION_THRESHOLD)
        persons.append(int((det.class_id == PERSON_CLASS_ID).sum()))
        vehicles.append(int(np.isin(det.class_id, list(VEHICLE_CLASS_IDS)).sum()))
        confs.append(float(det.confidence.mean()) if len(det) else 0.0)
    return dict(
        rfdetr_mean_person=float(np.mean(persons)), rfdetr_max_person=float(np.max(persons)),
        rfdetr_mean_vehicle=float(np.mean(vehicles)), rfdetr_mean_confidence=float(np.mean(confs)),
    )

## Step 4 — Per-clip extraction loop (resumable, branch-selective)

In [ ]:
DINOV2_DIM = 768
VIDEOMAE_DIM = 768

FIELDNAMES = ['video_id', 'label', 'split']
if 'lk' in BRANCHES:
    FIELDNAMES += ['lk_mean_magnitude', 'lk_max_magnitude', 'lk_mean_angle_deg', 'lk_mean_tracked']
if 'rfdetr' in BRANCHES:
    FIELDNAMES += ['rfdetr_mean_person', 'rfdetr_max_person', 'rfdetr_mean_vehicle', 'rfdetr_mean_confidence']
if 'dinov2' in BRANCHES:
    FIELDNAMES += [f'dinov2_{i}' for i in range(DINOV2_DIM)]
if 'videomae' in BRANCHES:
    FIELDNAMES += [f'videomae_{i}' for i in range(VIDEOMAE_DIM)]

already_done = set()
if os.path.exists(OUT_CSV):
    already_done = set(pd.read_csv(OUT_CSV, usecols=['video_id'])['video_id'])
    print(f"Resuming: {len(already_done)} clips already extracted for this branch selection.")

write_header = not os.path.exists(OUT_CSV)
with open(OUT_CSV, 'a', newline='') as fh:
    writer = csv.DictWriter(fh, fieldnames=FIELDNAMES)
    if write_header:
        writer.writeheader()

    for i, row in enumerate(all_clips.to_dict('records'), 1):
        if row['id'] in already_done:
            continue
        video_path = os.path.join(row['clip_dir'], row['id'])
        try:
            record = {'video_id': row['id'], 'label': row['label'], 'split': row['split']}

            if 'lk' in BRANCHES:
                record.update(compute_lk_summary(video_path))

            need_sparse = 'rfdetr' in BRANCHES or 'dinov2' in BRANCHES
            sparse_frames = sample_frames_rgb(video_path, N_SPARSE_FRAMES) if need_sparse else []

            if 'rfdetr' in BRANCHES:
                record.update(compute_rfdetr_summary(sparse_frames))

            if 'dinov2' in BRANCHES:
                dinov2_vec = (np.mean([dinov2_embed_frame(f) for f in sparse_frames], axis=0)
                              if sparse_frames else np.zeros(DINOV2_DIM))
                record.update({f'dinov2_{j}': float(v) for j, v in enumerate(dinov2_vec)})

            if 'videomae' in BRANCHES:
                videomae_frames = sample_frames_rgb(video_path, VIDEOMAE_NUM_FRAMES)
                if len(videomae_frames) < VIDEOMAE_NUM_FRAMES:
                    videomae_frames = (videomae_frames * VIDEOMAE_NUM_FRAMES)[:VIDEOMAE_NUM_FRAMES] if videomae_frames else []
                videomae_vec = videomae_embed_clip(videomae_frames) if videomae_frames else np.zeros(VIDEOMAE_DIM)
                record.update({f'videomae_{j}': float(v) for j, v in enumerate(videomae_vec)})

            writer.writerow(record)
            fh.flush()
        except Exception as e:
            print(f"  FAILED {row['id']}: {e}")
            continue

        if i % 25 == 0 or i == len(all_clips):
            print(f"  {i}/{len(all_clips)} clips processed")

print(f"\nDone. Features saved to {OUT_CSV}")

## Next steps

- Run this notebook once per branch group you need for the ablation:
  `{'lk', 'rfdetr'}` (cheap, CPU is fine) and `{'dinov2', 'videomae'}`
  (GPU strongly recommended), and optionally all four together to
  reproduce the original 92.27% full-fusion number.
- Feed whichever `features_*.csv` files you produce into
  `ablation_fusion_branches.ipynb` to compare them.
- If VideoMAE padding (repeating frames for short clips) shows up a lot
  in practice, worth checking clip-length distribution -- not checked
  yet in this notebook.